# Complete UNSW-NB15 Project Workflow

This notebook runs the project from start to finish: checks the real dataset, performs EDA, trains binary and multiclass models (including the SMOTE experiment), evaluates all five algorithms, runs predictions, and saves reports, charts, model artifacts, and prediction CSVs.

Training uses the official training CSV. The separate official testing CSV is held out for prediction and final evaluation. Running the training cells again will update the project model and report folders for these three real-data runs.

Run all cells from top to bottom with the project's Python environment selected as the Jupyter kernel.

## 1. Find the project and check the real datasets

The cell below finds the project folder and verifies that both official partitioned CSV files are present. Missing datasets produce an explicit error.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / 'src' / 'predict.py').is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / 'src' / 'predict.py').is_file():
    raise FileNotFoundError('Could not find the project root containing src/predict.py')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import joblib
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score,
                             f1_score, precision_score, recall_score)

from src import config as cfg
from src.predict import predict_frame

TRAIN_CSV = cfg.DEFAULT_TRAIN
TEST_CSV = cfg.DEFAULT_TEST
OUTPUT_DIR = ROOT / 'predictions'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUNS = {
    'Binary detection': ('binary_real', 'binary'),
    'Attack-family classification': ('multiclass_real', 'multiclass'),
    'Attack-family classification with SMOTE': ('multiclass_real_smote', 'multiclass'),
}
ALGORITHMS = {
    'Decision Tree': 'DecisionTree.joblib',
    'Logistic Regression': 'LogisticRegression.joblib',
    'Random Forest': 'RandomForest.joblib',
    'XGBoost': 'XGBoost.joblib',
    'Neural Network (MLP)': 'MLP.joblib',
}

required_data = [TRAIN_CSV, TEST_CSV]
missing_data = [path for path in required_data if not path.is_file()]
if missing_data:
    raise FileNotFoundError('Required UNSW-NB15 CSV files are missing:\n' + '\n'.join(map(str, missing_data)))

print(f'Project folder: {ROOT}')
print(f'Test dataset: {TEST_CSV}')
print(f'Training data: {TRAIN_CSV}')
print(f'Test data: {TEST_CSV}')

## 2. Explore the training dataset (EDA)

The project's EDA script checks data quality and class balance, then saves figures and summary tables under `reports/figures/eda/` and `reports/results/eda/`.

In [ ]:
from src.eda import main as run_eda

run_eda(['--train', str(TRAIN_CSV)])
print('EDA completed. Figures and summary tables are saved in reports/.')

## 3. Train the three real-data model runs

Each run compares Logistic Regression, Decision Tree, Random Forest, XGBoost, and MLP. Training uses only the training split; model evaluation uses the held-out test split. The third run applies SMOTE to the training data to study class imbalance. These full runs can take several minutes.

In [ ]:
from src.train import main as run_training

training_runs = [
    ('Binary detection', ['--task', 'binary', '--tag', 'real']),
    ('Attack-family classification', ['--task', 'multiclass', '--tag', 'real']),
    ('Attack-family classification with SMOTE', ['--task', 'multiclass', '--smote', '--tag', 'real_smote']),
]

for run_label, arguments in training_runs:
    print(f'\nStarting: {run_label}')
    run_training(['--train', str(TRAIN_CSV), '--test', str(TEST_CSV), *arguments])
    print(f'Finished: {run_label}')

print('\nAll training runs completed. Models, metrics, reports, and evaluation figures are saved under models/ and reports/.')

In [ ]:
required_models = [
    cfg.MODELS_DIR / folder / filename
    for folder, _ in RUNS.values()
    for filename in ALGORITHMS.values()
]
missing_models = [path for path in required_models if not path.is_file()]
if missing_models:
    raise FileNotFoundError('Training did not produce required model artifacts:\n' + '\n'.join(map(str, missing_models)))
print(f'Confirmed all {len(required_models)} model artifacts are available for prediction.')

## 4. Load and inspect the held-out test dataset

The input to each model is the flow-feature columns. The `label` and `attack_cat` columns are kept only as ground truth for evaluation; the prediction helper excludes them from model features.

In [ ]:
test_df = pd.read_csv(TEST_CSV)
required_targets = {cfg.LABEL_COL, cfg.ATTACK_COL}
if not required_targets.issubset(test_df.columns):
    raise ValueError(f'Test CSV must contain target columns {sorted(required_targets)}')

print(f'Dataset shape: {test_df.shape[0]:,} rows x {test_df.shape[1]} columns')
print('\nBinary ground-truth counts:')
print(test_df[cfg.LABEL_COL].map({0: 'Normal', 1: 'Attack'}).value_counts().to_string())
print('\nAttack-family ground-truth counts:')
print(test_df[cfg.ATTACK_COL].value_counts().to_string())
print('\nExample input rows:')
print(test_df.head(3).to_string(index=False))

## 5. Predict with every saved model

For each run and algorithm, the notebook loads the saved preprocessing-plus-model artifact and predicts every test row. Each algorithm's class prediction is added as a new column. The original test features and true labels stay in the output so rows can be traced and evaluated.

In [ ]:
all_predictions = test_df.copy()
metrics_rows = []
best_run_predictions = {}

true_binary = test_df[cfg.LABEL_COL].map({0: 'Normal', 1: 'Attack'}).astype(str)
true_multiclass = test_df[cfg.ATTACK_COL].astype(str).str.strip()

for run_label, (folder, task) in RUNS.items():
    y_true = true_binary if task == 'binary' else true_multiclass
    print(f'\n{run_label} ({folder})')
    for algorithm, filename in ALGORITHMS.items():
        model_path = cfg.MODELS_DIR / folder / filename
        artifact = joblib.load(model_path)
        predicted_frame, _ = predict_frame(artifact, test_df)
        y_pred = predicted_frame['predicted_class'].astype(str)
        prediction_column = f'{folder}__{algorithm}'
        all_predictions[prediction_column] = y_pred.to_numpy()

        classes = list(artifact['classes'])
        if not set(y_true.unique()).issubset(classes):
            raise ValueError(f'True labels do not match classes for {model_path}: {classes}')
        true_attack = y_true.to_numpy() != cfg.NORMAL_NAME
        predicted_attack = y_pred.to_numpy() != cfg.NORMAL_NAME
        attack_total = int(true_attack.sum())
        normal_total = int((~true_attack).sum())
        detection_rate = int((true_attack & predicted_attack).sum()) / attack_total if attack_total else 0.0
        false_alarm_rate = int((~true_attack & predicted_attack).sum()) / normal_total if normal_total else 0.0

        metrics_rows.append({
            'run': run_label,
            'algorithm': algorithm,
            'accuracy': accuracy_score(y_true, y_pred),
            'macro_f1': f1_score(y_true, y_pred, labels=classes, average='macro', zero_division=0),
            'weighted_f1': f1_score(y_true, y_pred, labels=classes, average='weighted', zero_division=0),
            'attack_detection_rate': detection_rate,
            'false_alarm_rate': false_alarm_rate,
        })
        print(f'  {algorithm}: predictions complete')

        if algorithm == 'XGBoost' and task in ('binary', 'multiclass'):
            best_run_predictions[run_label] = (y_true.copy(), y_pred.copy(), classes)

        del artifact, predicted_frame, y_pred

metrics_df = pd.DataFrame(metrics_rows)
print(f'\nCompleted predictions from {len(metrics_df)} saved model artifacts on {len(test_df):,} test rows each.')

## 6. Compare model results

Accuracy measures the overall fraction correct. Macro-F1 gives each class equal importance, which helps show performance on rare attack families. Attack detection rate measures what fraction of actual attacks were flagged, while false-alarm rate measures how often normal flows were incorrectly flagged.

In [ ]:
display_metrics = metrics_df.sort_values(['run', 'macro_f1'], ascending=[True, False]).reset_index(drop=True)
print(display_metrics.to_string(index=False, float_format=lambda value: f'{value:.4f}'))

metrics_path = OUTPUT_DIR / 'all_model_metrics.csv'
display_metrics.to_csv(metrics_path, index=False)
print(f'\nSaved metric table: {metrics_path}')

## 7. Save and inspect all predictions

The combined file contains every original test row plus one predicted-class column for each of the 15 model/run combinations. Use it to inspect an individual flow and compare what the different algorithms predicted.

In [ ]:
predictions_path = OUTPUT_DIR / 'all_model_predictions.csv'
all_predictions.to_csv(predictions_path, index=False)
print(f'Saved all predictions: {predictions_path}')
print(f'Output shape: {all_predictions.shape[0]:,} rows x {all_predictions.shape[1]} columns')

prediction_columns = [column for column in all_predictions.columns if '__' in column]
print('\nFirst 10 rows of predictions from every model:')
print(all_predictions[prediction_columns].head(10).to_string(index=False))

print('\nPredicted class counts for each model:')
for column in prediction_columns:
    print(f'\n{column}')
    print(all_predictions[column].value_counts().to_string())

## 8. Visualize the metric comparison

These charts compare the 5 algorithms within each run. They make it easier to explain how baseline multiclass training, SMOTE, and the binary task differ.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
for axis, (run_label, _) in zip(axes, RUNS.items()):
    run_metrics = metrics_df[metrics_df['run'] == run_label].set_index('algorithm')
    run_metrics[['accuracy', 'macro_f1', 'attack_detection_rate']].plot.bar(
        ax=axis, ylim=(0, 1), rot=35, title=run_label,
    )
    axis.set_ylabel('Score')
    axis.grid(axis='y', alpha=0.25)
fig.tight_layout()
comparison_path = OUTPUT_DIR / 'all_model_metric_comparison.png'
fig.savefig(comparison_path, dpi=160, bbox_inches='tight')
plt.show()
print(f'Saved metric comparison chart: {comparison_path}')

## 9. View XGBoost confusion matrices

The confusion matrix shows true classes on one axis and predicted classes on the other. The cells are row-normalized, so each row shows the proportion of a true class predicted as each class. XGBoost is shown here as a representative model; the full metric table above compares all algorithms.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for axis, run_label in zip(axes, ['Binary detection', 'Attack-family classification']):
    y_true, y_pred, classes = best_run_predictions[run_label]
    ConfusionMatrixDisplay.from_predictions(
        y_true, y_pred, labels=classes, display_labels=classes,
        normalize='true', values_format='.2f', cmap='Blues', ax=axis, colorbar=False,
    )
    axis.set_title(f'{run_label} — XGBoost')
    axis.tick_params(axis='x', labelrotation=45)
fig.tight_layout()
confusion_path = OUTPUT_DIR / 'xgboost_confusion_matrices.png'
fig.savefig(confusion_path, dpi=160, bbox_inches='tight')
plt.show()
print(f'Saved confusion matrices: {confusion_path}')

## Explanation you can present

1. EDA explores the training dataset and saves data-quality summaries and charts.
2. The three training runs fit preprocessing and five model algorithms on training data only; SMOTE is limited to the third run's training data.
3. Each saved pipeline handles feature preprocessing and predicts a class for every held-out test flow.
4. It evaluates 15 model/run combinations: 5 algorithms for binary detection, 5 for multiclass classification, and 5 for multiclass classification trained with SMOTE.
5. It compares predictions with the known test labels and reports accuracy, macro-F1, weighted-F1, attack detection rate, and false-alarm rate.
6. It saves trained models and detailed reports under `models/` and `reports/`, and combined predictions, metrics, and summary charts under `predictions/`.

**Output files:** `all_model_predictions.csv`, `all_model_metrics.csv`, `all_model_metric_comparison.png`, and `xgboost_confusion_matrices.png`.